# Module 3: Production Deployment with Amazon Bedrock AgentCore

## Overview

This module deploys the e-commerce multi-agent system to production using **Amazon Bedrock AgentCore**:

1. **Lambda MCP Tools**: Deploy Lambda functions as MCP tool backends
2. **AgentCore Gateway**: Expose Lambda tools via MCP protocol
3. **Cognito Authentication**: Secure Gateway access with JWT tokens
4. **AgentCore Runtime**: Deploy specialized agents with A2A communication
5. **Orchestrator Agent**: Main entry point that routes to specialized agents
6. **Streamlit Frontend**: Interactive chat UI connected to the Orchestrator

### Architecture

```
                                    ┌─────────────────────────────┐
                                    │   AgentCore Runtime         │
                                    │  ┌─────────────────────────┐│
                                    │  │  Orchestrator Agent     ││
                                    │  │  (Main Entry Point)     ││
                                    │  └───────────┬─────────────┘│
                                    │              │ A2A          │
                                    │    ┌─────────┼─────────┐    │
                                    │    │         │         │    │
                                    │  ┌─▼───┐  ┌──▼──┐  ┌───▼─┐  │
                                    │  │Order│  │Prod │  │Acct │  │
                                    │  │Agent│  │Agent│  │Agent│  │
                                    │  └──┬──┘  └──┬──┘  └──┬──┘  │
                                    │     │        │        │     │
                                    └─────┼────────┼────────┼─────┘
                                          │   MCP  │        │
                                          └────────┼────────┘
                                                   │
                                    ┌──────────────▼──────────────┐
User ──► Streamlit ──► A2A ──►     │    AgentCore Gateway        │
                                    │  ┌───────┬───────┬───────┐  │
                                    │  │Order  │Product│Account│  │
                                    │  │Tools  │Tools  │Tools  │  │
                                    │  └───┬───┴───┬───┴───┬───┘  │
                                    └──────┼───────┼───────┼──────┘
                                           │Lambda │       │
                                    ┌──────▼───────▼───────▼──────┐
                                    │      DynamoDB Tables        │
                                    │  Orders │ Products │Accounts│
                                    └─────────────────────────────┘
```

### Prerequisites

- AWS credentials with AgentCore, Lambda, Cognito, ECR, and DynamoDB permissions
- Module 1 DynamoDB tables created
- Python 3.10+
- Docker (for building container images)

### Time: ~45 minutes

## Step 1: Setup and Configuration

In [ ]:
# Install dependencies
!pip install -q boto3 strands-agents mcp requests

In [ ]:
import boto3
import json
import os
import time

# Try to load REGION from previous modules
try:
    %store -r REGION
    print(f"Loaded REGION from previous module: {REGION}")
except:
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'
    print(f"Using default region: {REGION}")

os.environ['AWS_REGION'] = REGION
os.environ['AWS_DEFAULT_REGION'] = REGION

# Workshop configuration
WORKSHOP_PREFIX = 'ecommerce-workshop'
GATEWAY_NAME = f'{WORKSHOP_PREFIX}-gateway'
LAMBDA_ROLE_NAME = f'{WORKSHOP_PREFIX}-lambda-role'
GATEWAY_ROLE_NAME = f'{WORKSHOP_PREFIX}-gateway-role'
COGNITO_POOL_NAME = f'{WORKSHOP_PREFIX}-user-pool'
COGNITO_RESOURCE_SERVER_ID = f'{WORKSHOP_PREFIX}-gateway-api'

# DynamoDB table names (from Module 1)
ORDERS_TABLE = f'{WORKSHOP_PREFIX}-orders'
PRODUCTS_TABLE = f'{WORKSHOP_PREFIX}-products'
ACCOUNTS_TABLE = f'{WORKSHOP_PREFIX}-accounts'

print(f"\nConfiguration:")
print(f"  Region: {REGION}")
print(f"  Gateway: {GATEWAY_NAME}")
print(f"  Tables: {ORDERS_TABLE}, {PRODUCTS_TABLE}, {ACCOUNTS_TABLE}")

In [ ]:
# Verify AWS credentials and get account ID
sts = boto3.client('sts', region_name=REGION)
identity = sts.get_caller_identity()
ACCOUNT_ID = identity['Account']

print(f"AWS Account: {ACCOUNT_ID}")
print(f"AWS User/Role: {identity['Arn']}")

## Step 2: Create IAM Roles

In [ ]:
from utils import create_lambda_execution_role, create_agentcore_gateway_role

iam_client = boto3.client('iam', region_name=REGION)

# DynamoDB table ARNs
dynamodb_table_arns = [
    f'arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{ORDERS_TABLE}',
    f'arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{PRODUCTS_TABLE}',
    f'arn:aws:dynamodb:{REGION}:{ACCOUNT_ID}:table/{ACCOUNTS_TABLE}'
]

# Create Lambda execution role
print("Creating Lambda execution role...")
lambda_role_resp = create_lambda_execution_role(
    iam_client,
    LAMBDA_ROLE_NAME,
    dynamodb_table_arns
)
LAMBDA_ROLE_ARN = lambda_role_resp['Role']['Arn']
print(f"Lambda Role ARN: {LAMBDA_ROLE_ARN}")

## Step 3: Deploy Lambda MCP Tool Functions

In [ ]:
from utils import create_lambda_function

lambda_client = boto3.client('lambda', region_name=REGION)

# Environment variables for all Lambda functions
# Note: AWS_REGION is a reserved Lambda variable - it's automatically set by AWS
lambda_env_vars = {
    'ORDERS_TABLE_NAME': ORDERS_TABLE,
    'PRODUCTS_TABLE_NAME': PRODUCTS_TABLE,
    'ACCOUNTS_TABLE_NAME': ACCOUNTS_TABLE
}

# Lambda function configurations
lambda_configs = [
    {
        'name': f'{WORKSHOP_PREFIX}-order-tools',
        'code_path': 'lambda_tools/order_tools_lambda.py',
        'handler': 'order_tools_lambda.lambda_handler'
    },
    {
        'name': f'{WORKSHOP_PREFIX}-product-tools',
        'code_path': 'lambda_tools/product_tools_lambda.py',
        'handler': 'product_tools_lambda.lambda_handler'
    },
    {
        'name': f'{WORKSHOP_PREFIX}-account-tools',
        'code_path': 'lambda_tools/account_tools_lambda.py',
        'handler': 'account_tools_lambda.lambda_handler'
    }
]

lambda_arns = {}

print("Deploying Lambda functions...\n")
for config in lambda_configs:
    print(f"Deploying {config['name']}...")
    result = create_lambda_function(
        lambda_client,
        config['name'],
        LAMBDA_ROLE_ARN,
        config['code_path'],
        config['handler'],
        lambda_env_vars,
        REGION
    )
    if result['exit_code'] == 0:
        lambda_arns[config['name']] = result['function_arn']
        print(f"  ARN: {result['function_arn']}\n")
    else:
        print(f"  ERROR: {result.get('error')}\n")

print(f"\nDeployed {len(lambda_arns)} Lambda functions")

## Step 4: Create Cognito User Pool for Authentication

In [ ]:
from utils import (
    get_or_create_cognito_user_pool,
    get_or_create_cognito_resource_server,
    get_or_create_cognito_app_client,
    get_or_create_cognito_domain,
    create_test_user
)

cognito_client = boto3.client('cognito-idp', region_name=REGION)

# Create User Pool
print("Setting up Cognito authentication...\n")
USER_POOL_ID = get_or_create_cognito_user_pool(cognito_client, COGNITO_POOL_NAME)

# Create Resource Server with custom scopes
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access to Gateway tools"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access to Gateway tools"}
]
get_or_create_cognito_resource_server(
    cognito_client,
    USER_POOL_ID,
    COGNITO_RESOURCE_SERVER_ID,
    f"{WORKSHOP_PREFIX} Gateway API",
    SCOPES
)

# Create App Client for M2M authentication
CLIENT_NAME = f'{WORKSHOP_PREFIX}-gateway-client'
CLIENT_ID, CLIENT_SECRET = get_or_create_cognito_app_client(
    cognito_client,
    USER_POOL_ID,
    CLIENT_NAME,
    COGNITO_RESOURCE_SERVER_ID
)

# Create domain for OAuth endpoints
import uuid
DOMAIN_PREFIX = f"{WORKSHOP_PREFIX}-{str(uuid.uuid4())[:8]}"
get_or_create_cognito_domain(cognito_client, USER_POOL_ID, DOMAIN_PREFIX)

# Create test user
TEST_USER_EMAIL = 'testuser@ecommerce-workshop.example'
TEST_USER_PASSWORD = 'TestPass123!'
create_test_user(cognito_client, USER_POOL_ID, CLIENT_ID, TEST_USER_EMAIL, TEST_USER_PASSWORD)

# Get discovery URL
COGNITO_DISCOVERY_URL = f'https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration'

print(f"\nCognito Configuration:")
print(f"  User Pool ID: {USER_POOL_ID}")
print(f"  Client ID: {CLIENT_ID}")
print(f"  Domain: {DOMAIN_PREFIX}")
print(f"  Discovery URL: {COGNITO_DISCOVERY_URL}")

## Step 5: Create AgentCore Gateway Role

In [ ]:
# Create Gateway IAM role with Lambda invoke permissions
print("Creating AgentCore Gateway role...")
gateway_role_resp = create_agentcore_gateway_role(
    iam_client,
    GATEWAY_ROLE_NAME,
    list(lambda_arns.values())
)
GATEWAY_ROLE_ARN = gateway_role_resp['Role']['Arn']
print(f"Gateway Role ARN: {GATEWAY_ROLE_ARN}")

## Step 6: Create AgentCore Gateway

In [ ]:
from utils import create_gateway

gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Cognito authorizer configuration
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [CLIENT_ID],
        "discoveryUrl": COGNITO_DISCOVERY_URL
    }
}

print("Creating AgentCore Gateway...")
gateway_response = create_gateway(
    gateway_client,
    GATEWAY_NAME,
    GATEWAY_ROLE_ARN,
    auth_config,
    'E-Commerce Customer Service MCP Gateway'
)

if gateway_response:
    GATEWAY_ID = gateway_response['gatewayId']
    GATEWAY_URL = gateway_response['gatewayUrl']
    print(f"\nGateway Created:")
    print(f"  Gateway ID: {GATEWAY_ID}")
    print(f"  Gateway URL: {GATEWAY_URL}")
else:
    print("Gateway creation failed - check if it already exists")

## Step 7: Create Lambda Gateway Targets (MCP Tools)

In [ ]:
from utils import create_lambda_gateway_target

# Define tool schemas for Order Tools
ORDER_TOOL_SCHEMAS = [
    {
        "name": "get_order_status",
        "description": "Get the current status and details of an order by order ID",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID to look up"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "track_shipment",
        "description": "Get tracking information for a shipped order",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID to track"}
            },
            "required": ["order_id"]
        }
    },
    {
        "name": "process_return",
        "description": "Initiate a return request for an order",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID to return"},
                "reason": {"type": "string", "description": "Reason for the return"}
            },
            "required": ["order_id", "reason"]
        }
    },
    {
        "name": "modify_order",
        "description": "Modify an order (cancel or update shipping address)",
        "inputSchema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID to modify"},
                "modification_type": {"type": "string", "description": "Type: cancel or shipping_address"},
                "new_value": {"type": "string", "description": "New value for modification"}
            },
            "required": ["order_id", "modification_type", "new_value"]
        }
    },
    {
        "name": "get_customer_orders",
        "description": "Get order history for a customer by email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "customer_email": {"type": "string", "description": "Customer email address"},
                "limit": {"type": "integer", "description": "Maximum orders to return"}
            },
            "required": ["customer_email"]
        }
    }
]

# Define tool schemas for Product Tools
PRODUCT_TOOL_SCHEMAS = [
    {
        "name": "search_products",
        "description": "Search for products in the catalog using keywords",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search keywords"},
                "category": {"type": "string", "description": "Product category filter"},
                "max_results": {"type": "integer", "description": "Maximum results to return"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_product_details",
        "description": "Get detailed information about a specific product",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "Product ID to look up"}
            },
            "required": ["product_id"]
        }
    },
    {
        "name": "check_inventory",
        "description": "Check inventory availability for a product",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "Product ID to check"}
            },
            "required": ["product_id"]
        }
    },
    {
        "name": "get_product_recommendations",
        "description": "Get product recommendations based on criteria",
        "inputSchema": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "description": "Product category"},
                "price_max": {"type": "number", "description": "Maximum price filter"},
                "limit": {"type": "integer", "description": "Number of recommendations"}
            }
        }
    },
    {
        "name": "compare_products",
        "description": "Compare multiple products side by side",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_ids": {"type": "array", "items": {"type": "string"}, "description": "List of product IDs to compare"}
            },
            "required": ["product_ids"]
        }
    },
    {
        "name": "get_return_policy",
        "description": "Get return policy information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "product_id": {"type": "string", "description": "Optional product ID for specific policy"}
            }
        }
    }
]

# Define tool schemas for Account Tools
ACCOUNT_TOOL_SCHEMAS = [
    {
        "name": "get_account_info",
        "description": "Get customer account information by email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "customer_email": {"type": "string", "description": "Customer email address"}
            },
            "required": ["customer_email"]
        }
    },
    {
        "name": "update_shipping_address",
        "description": "Update customer's default shipping address",
        "inputSchema": {
            "type": "object",
            "properties": {
                "customer_email": {"type": "string", "description": "Customer email"},
                "address": {"type": "object", "description": "New address object"}
            },
            "required": ["customer_email", "address"]
        }
    },
    {
        "name": "get_membership_benefits",
        "description": "Get membership tier benefits and comparison",
        "inputSchema": {
            "type": "object",
            "properties": {
                "tier": {"type": "string", "description": "Membership tier: standard, gold, or platinum"}
            },
            "required": ["tier"]
        }
    },
    {
        "name": "initiate_password_reset",
        "description": "Initiate a password reset request",
        "inputSchema": {
            "type": "object",
            "properties": {
                "customer_email": {"type": "string", "description": "Customer email"}
            },
            "required": ["customer_email"]
        }
    },
    {
        "name": "update_notification_preferences",
        "description": "Update customer notification preferences",
        "inputSchema": {
            "type": "object",
            "properties": {
                "customer_email": {"type": "string", "description": "Customer email"},
                "preferences": {"type": "object", "description": "Notification preferences"}
            },
            "required": ["customer_email", "preferences"]
        }
    },
    {
        "name": "get_reward_points",
        "description": "Get customer's reward points balance and history",
        "inputSchema": {
            "type": "object",
            "properties": {
                "customer_email": {"type": "string", "description": "Customer email"}
            },
            "required": ["customer_email"]
        }
    }
]

print("Tool schemas defined for Order, Product, and Account tools")

In [ ]:
# Create Order Tools target
print("Creating Gateway Lambda targets...\n")

order_target = create_lambda_gateway_target(
    gateway_client,
    GATEWAY_ID,
    'OrderTools',
    lambda_arns[f'{WORKSHOP_PREFIX}-order-tools'],
    ORDER_TOOL_SCHEMAS,
    'Order management tools - status, tracking, returns, modifications'
)
print(f"Order Tools target created\n")

# Create Product Tools target
product_target = create_lambda_gateway_target(
    gateway_client,
    GATEWAY_ID,
    'ProductTools',
    lambda_arns[f'{WORKSHOP_PREFIX}-product-tools'],
    PRODUCT_TOOL_SCHEMAS,
    'Product catalog tools - search, details, inventory, recommendations'
)
print(f"Product Tools target created\n")

# Create Account Tools target
account_target = create_lambda_gateway_target(
    gateway_client,
    GATEWAY_ID,
    'AccountTools',
    lambda_arns[f'{WORKSHOP_PREFIX}-account-tools'],
    ACCOUNT_TOOL_SCHEMAS,
    'Account management tools - info, address, membership, rewards'
)
print(f"Account Tools target created\n")

print("All Gateway targets created!")

## Step 8: Save Gateway Configuration

In [ ]:
from utils import save_gateway_config

# Save configuration for Streamlit app
gateway_config = {
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'region': REGION,
    'auth_type': 'JWT',
    'client_info': {
        'user_pool_id': USER_POOL_ID,
        'client_id': CLIENT_ID,
        'username': TEST_USER_EMAIL,
        'password': TEST_USER_PASSWORD
    },
    'model_id': 'global.anthropic.claude-3-5-sonnet-20241022-v2:0'
}

# Save to streamlit_app directory
save_gateway_config(gateway_config, 'streamlit_app/gateway_config.json')

print("\nGateway configuration saved for Streamlit app")

## Step 9: Test Gateway Connection

In [ ]:
# Wait for resources to propagate
print("Waiting for Gateway to be ready...")
time.sleep(10)

In [ ]:
# Test OAuth token retrieval
from utils import get_oauth_token

SCOPE_STRING = f"{COGNITO_RESOURCE_SERVER_ID}/gateway:read {COGNITO_RESOURCE_SERVER_ID}/gateway:write"

print("Getting OAuth token from Cognito...")
token_response = get_oauth_token(
    USER_POOL_ID,
    CLIENT_ID,
    CLIENT_SECRET,
    SCOPE_STRING,
    REGION
)

if 'access_token' in token_response:
    ACCESS_TOKEN = token_response['access_token']
    print(f"Access token obtained (expires in {token_response.get('expires_in', 'unknown')}s)")
    print(f"Token preview: {ACCESS_TOKEN[:50]}...")
else:
    print(f"Failed to get token: {token_response}")

In [ ]:
# Test MCP connection to Gateway
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("Testing MCP connection to Gateway...\n")

def create_mcp_transport():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {ACCESS_TOKEN}"}
    )

mcp_client = MCPClient(create_mcp_transport)

with mcp_client:
    # List available tools
    tools = mcp_client.list_tools_sync()
    
    print(f"Found {len(tools)} MCP tools:\n")
    for tool in tools:
        print(f"  - {tool.tool_name}")

print("\nGateway connection successful!")

## Step 10: Test Agent with Gateway Tools

In [ ]:
from strands import Agent
from strands.models import BedrockModel

print("Creating Strands Agent with Gateway tools...\n")

# Create model
model = BedrockModel(
    model_id='global.anthropic.claude-sonnet-4-20250514-v1:0',
    region_name=REGION,
    temperature=0.2
)

# System prompt
system_prompt = """You are a helpful e-commerce customer service agent.
Use the available tools to help customers with orders, products, and account inquiries.
Always be polite and provide accurate information from the tools."""

# Create agent with MCP tools
with mcp_client:
    tools = mcp_client.list_tools_sync()
    agent = Agent(
        model=model,
        tools=tools,
        system_prompt=system_prompt
    )
    
    print(f"Agent created with {len(agent.tool_names)} tools")
    print(f"Tools: {agent.tool_names}\n")
    
    # Test query
    print("Testing agent with order status query...\n")
    response = agent("What's the status of order ORD-2024-10002?")
    print(f"Response:\n{response}")

In [ ]:
# Test more queries
with mcp_client:
    tools = mcp_client.list_tools_sync()
    agent = Agent(
        model=model,
        tools=tools,
        system_prompt=system_prompt
    )
    
    # print("Test 2: Product search\n")
    # response = agent("Do you have any wireless headphones under $100?")
    # print(f"Response:\n{response}\n")
    # print("="*50)
    
    print("\nTest 3: Membership benefits\n")
    response = agent("What are the benefits of Gold membership?")
    print(f"Response:\n{response}")

## Part 2: Deploy Multi-Agent System to AgentCore Runtime

Now that the Gateway with MCP tools is ready, we'll deploy specialized agents to AgentCore Runtime using A2A (Agent-to-Agent) protocol for inter-agent communication.

### Architecture Overview
- **Order Agent**: Handles order status, tracking, returns, modifications
- **Product Agent**: Handles product search, details, inventory, recommendations
- **Account Agent**: Handles account info, addresses, membership, rewards
- **Orchestrator Agent**: Routes requests to appropriate specialized agents via A2A

In [ ]:
print("="*60)
print("DEPLOYMENT COMPLETE!")
print("="*60)
print(f"\nGateway URL: {GATEWAY_URL}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"\nMCP Tools: {len(ORDER_TOOL_SCHEMAS) + len(PRODUCT_TOOL_SCHEMAS) + len(ACCOUNT_TOOL_SCHEMAS)} total")
print(f"  - Order Tools: {len(ORDER_TOOL_SCHEMAS)}")
print(f"  - Product Tools: {len(PRODUCT_TOOL_SCHEMAS)}")
print(f"  - Account Tools: {len(ACCOUNT_TOOL_SCHEMAS)}")
print(f"\nTo launch the Streamlit app:")
print(f"  cd streamlit_app")
print(f"  pip install streamlit")
print(f"  streamlit run app.py")
print(f"\nThe app will be available at http://localhost:8501")
print("="*60)

In [ ]:
# Save deployment info for Module 4
deployment_info = {
    'gateway_id': GATEWAY_ID,
    'gateway_url': GATEWAY_URL,
    'region': REGION,
    'lambda_arns': lambda_arns,
    'user_pool_id': USER_POOL_ID,
    'client_id': CLIENT_ID
}

%store deployment_info
%store REGION
print("Deployment information saved for subsequent modules")



## Part 1 Complete: Gateway and MCP Tools Deployed

You have successfully deployed the MCP tools to AgentCore Gateway.

### What Was Created (Part 1)

| Component | Description |
|-----------|-------------|
| Lambda Functions | 3 functions (order, product, account tools) |
| Cognito User Pool | JWT authentication for Gateway access |
| AgentCore Gateway | MCP protocol endpoint with 17 tools |

Continue to Part 2 to deploy the multi-agent system to AgentCore Runtime.

---

## Part 2

## Step 12: Create ECR Repository and Build Agent Container Images

AgentCore Runtime requires containerized agents. We'll build Docker images for each specialized agent.

In [ ]:
# Create ECR repositories for agent containers
ecr_client = boto3.client('ecr', region_name=REGION)

AGENT_REPOS = [
    f'{WORKSHOP_PREFIX}-order-agent',
    f'{WORKSHOP_PREFIX}-product-agent',
    f'{WORKSHOP_PREFIX}-account-agent',
    f'{WORKSHOP_PREFIX}-orchestrator-agent'
]

print("Creating ECR repositories for agents...\n")
for repo_name in AGENT_REPOS:
    try:
        ecr_client.create_repository(
            repositoryName=repo_name,
            imageScanningConfiguration={'scanOnPush': True},
            imageTagMutability='MUTABLE'
        )
        print(f"Created: {repo_name}")
    except ecr_client.exceptions.RepositoryAlreadyExistsException:
        print(f"Exists: {repo_name}")

# Get ECR login command
ecr_auth = ecr_client.get_authorization_token()
ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
print(f"\nECR Registry: {ECR_REGISTRY}")

In [ ]:
# Build and push Docker images for each agent
import subprocess

# Login to ECR
print("Logging into ECR...")
login_cmd = f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {ECR_REGISTRY}"
subprocess.run(login_cmd, shell=True, check=True)

# Agent configurations
AGENTS = [
    {'name': 'order-agent', 'file': 'order_agent.py'},
    {'name': 'product-agent', 'file': 'product_agent.py'},
    {'name': 'account-agent', 'file': 'account_agent.py'},
    {'name': 'orchestrator-agent', 'file': 'orchestrator_agent.py'}
]

CONTAINER_URIS = {}

print("\nBuilding and pushing agent containers...\n")
for agent in AGENTS:
    repo_name = f"{WORKSHOP_PREFIX}-{agent['name']}"
    image_uri = f"{ECR_REGISTRY}/{repo_name}:latest"
    
    print(f"Building {agent['name']}...")
    
    # Build the image for ARM64 (required by AgentCore Runtime)
    build_cmd = f"docker build -t {repo_name} --build-arg AGENT_FILE={agent['file']} --platform linux/arm64 agents/"
    result = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  Build failed: {result.stderr}")
        continue
    
    # Tag for ECR
    tag_cmd = f"docker tag {repo_name}:latest {image_uri}"
    subprocess.run(tag_cmd, shell=True, check=True)
    
    # Push to ECR
    print(f"  Pushing to ECR...")
    push_cmd = f"docker push {image_uri}"
    result = subprocess.run(push_cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  Push failed: {result.stderr}")
        continue
    
    CONTAINER_URIS[agent['name']] = image_uri
    print(f"  Done: {image_uri}\n")

print(f"Built {len(CONTAINER_URIS)} agent containers")

## Step 13: Create AgentCore Runtime IAM Role

In [ ]:
from utils import create_agent_runtime_role

RUNTIME_ROLE_NAME = f'{WORKSHOP_PREFIX}-runtime-role'

# Create Runtime IAM role with Bedrock and DynamoDB permissions
print("Creating AgentCore Runtime role...")
runtime_role_resp = create_agent_runtime_role(
    iam_client,
    RUNTIME_ROLE_NAME,
    dynamodb_table_arns
)
RUNTIME_ROLE_ARN = runtime_role_resp['Role']['Arn']
print(f"Runtime Role ARN: {RUNTIME_ROLE_ARN}")

# Add additional permissions for the role
additional_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "ecr:GetDownloadUrlForLayer",
                "ecr:BatchGetImage",
                "ecr:BatchCheckLayerAvailability",
                "ecr:GetAuthorizationToken"
            ],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "cognito-idp:DescribeUserPool",
                "cognito-idp:DescribeUserPoolClient"
            ],
            "Resource": f"arn:aws:cognito-idp:{REGION}:*:userpool/*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:InvokeAgentRuntime"
            ],
            "Resource": "*"
        }
    ]
}

try:
    iam_client.put_role_policy(
        RoleName=RUNTIME_ROLE_NAME,
        PolicyName=f"{RUNTIME_ROLE_NAME}-additional-policy",
        PolicyDocument=json.dumps(additional_policy)
    )
    print("Added ECR, CloudWatch Logs, Cognito, and AgentCore permissions")
except Exception as e:
    print(f"Policy update note: {e}")

## Step 14: Deploy Specialized Agents to AgentCore Runtime

Deploy the Order, Product, and Account agents first, then the Orchestrator.

In [ ]:
from utils import create_agent_runtime

agentcore_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

# Common environment variables for specialized agents
SPECIALIZED_AGENT_ENV = {
    'AGENT_REGION': REGION,
    'GATEWAY_URL': GATEWAY_URL,
    'USER_POOL_ID': USER_POOL_ID,
    'CLIENT_ID': CLIENT_ID,
    'CLIENT_SECRET': CLIENT_SECRET,
    'MODEL_ID': 'global.anthropic.claude-haiku-4-5-20251001-v1:0'
}

print("="*60)
print("Deploying Order Agent to AgentCore Runtime...")
print("="*60)

# First, try to find existing runtime
ORDER_AGENT_ARN = None
ORDER_RUNTIME = None
runtime_name = 'ecommerce_workshop_order_agent_notrace'
try:
    response = agentcore_client.list_agent_runtimes()
    # Use 'agentRuntimes' key instead of 'items'
    for runtime in response.get('agentRuntimes', []):
        if runtime.get('agentRuntimeName') == runtime_name:
            print(f"Found existing Order Agent runtime: {runtime['agentRuntimeId']}")
            ORDER_RUNTIME = {
                'agentRuntimeId': runtime['agentRuntimeId'],
                'agentRuntimeArn': runtime['agentRuntimeArn'],
                'status': runtime.get('status')
            }
            ORDER_AGENT_ARN = runtime['agentRuntimeArn']
            print(f"Using existing Order Agent ARN: {ORDER_AGENT_ARN}")
            break
except Exception as e:
    print(f"Error checking for existing runtime: {e}")

# If no existing runtime found, create new one
if not ORDER_RUNTIME:
    ORDER_RUNTIME = create_agent_runtime(
        agentcore_client,
        runtime_name=runtime_name,
        role_arn=RUNTIME_ROLE_ARN,
        container_uri=CONTAINER_URIS.get('order-agent'),
        environment_vars=SPECIALIZED_AGENT_ENV,
        protocol_type='HTTP',
        description='Order management agent - handles order status, tracking, returns, modifications',
        network_mode='PUBLIC'
    )
    
    if ORDER_RUNTIME:
        ORDER_AGENT_ARN = ORDER_RUNTIME['agentRuntimeArn']
        print(f"Created new Order Agent Runtime ID: {ORDER_RUNTIME['agentRuntimeId']}")
        print(f"Order Agent ARN: {ORDER_AGENT_ARN}")
    else:
        print("Failed to create Order Agent runtime")

In [ ]:
# Deploy Product Agent
print("="*60)
print("Deploying Product Agent to AgentCore Runtime...")
print("="*60)

# First, try to find existing runtime
PRODUCT_AGENT_ARN = None
PRODUCT_RUNTIME = None
runtime_name = 'ecommerce_workshop_product_agent_notrace'
try:
    response = agentcore_client.list_agent_runtimes()
    # Use 'agentRuntimes' key instead of 'items'
    for runtime in response.get('agentRuntimes', []):
        if runtime.get('agentRuntimeName') == runtime_name:
            print(f"Found existing Product Agent runtime: {runtime['agentRuntimeId']}")
            PRODUCT_RUNTIME = {
                'agentRuntimeId': runtime['agentRuntimeId'],
                'agentRuntimeArn': runtime['agentRuntimeArn'],
                'status': runtime.get('status')
            }
            PRODUCT_AGENT_ARN = runtime['agentRuntimeArn']
            print(f"Using existing Product Agent ARN: {PRODUCT_AGENT_ARN}")
            break
except Exception as e:
    print(f"Error checking for existing runtime: {e}")

# If no existing runtime found, create new one
if not PRODUCT_RUNTIME:
    PRODUCT_RUNTIME = create_agent_runtime(
        agentcore_client,
        runtime_name=runtime_name,
        role_arn=RUNTIME_ROLE_ARN,
        container_uri=CONTAINER_URIS.get('product-agent'),
        environment_vars=SPECIALIZED_AGENT_ENV,
        protocol_type='HTTP',
        description='Product catalog agent - handles search, details, inventory, recommendations',
        network_mode='PUBLIC'
    )
    
    if PRODUCT_RUNTIME:
        PRODUCT_AGENT_ARN = PRODUCT_RUNTIME['agentRuntimeArn']
        print(f"Created new Product Agent Runtime ID: {PRODUCT_RUNTIME['agentRuntimeId']}")
        print(f"Product Agent ARN: {PRODUCT_AGENT_ARN}")
    else:
        print("Failed to create Product Agent runtime")

In [ ]:
# Deploy Account Agent
print("="*60)
print("Deploying Account Agent to AgentCore Runtime...")
print("="*60)

# First, try to find existing runtime
ACCOUNT_AGENT_ARN = None
ACCOUNT_RUNTIME = None
runtime_name='ecommerce_workshop_account_agent_notrace'
try:
    response = agentcore_client.list_agent_runtimes()
    # Use 'agentRuntimes' key instead of 'items'
    for runtime in response.get('agentRuntimes', []):
        if runtime.get('agentRuntimeName') == runtime_name:
            print(f"Found existing Account Agent runtime: {runtime['agentRuntimeId']}")
            ACCOUNT_RUNTIME = {
                'agentRuntimeId': runtime['agentRuntimeId'],
                'agentRuntimeArn': runtime['agentRuntimeArn'],
                'status': runtime.get('status')
            }
            ACCOUNT_AGENT_ARN = runtime['agentRuntimeArn']
            print(f"Using existing Account Agent ARN: {ACCOUNT_AGENT_ARN}")
            break
except Exception as e:
    print(f"Error checking for existing runtime: {e}")

# If no existing runtime found, create new one
if not ACCOUNT_RUNTIME:
    ACCOUNT_RUNTIME = create_agent_runtime(
        agentcore_client,
        runtime_name=runtime_name,
        role_arn=RUNTIME_ROLE_ARN,
        container_uri=CONTAINER_URIS.get('account-agent'),
        environment_vars=SPECIALIZED_AGENT_ENV,
        protocol_type='HTTP',
        description='Account management agent - handles account info, addresses, membership, rewards',
        network_mode='PUBLIC'
    )
    
    if ACCOUNT_RUNTIME:
        ACCOUNT_AGENT_ARN = ACCOUNT_RUNTIME['agentRuntimeArn']
        print(f"Created new Account Agent Runtime ID: {ACCOUNT_RUNTIME['agentRuntimeId']}")
        print(f"Account Agent ARN: {ACCOUNT_AGENT_ARN}")
    else:
        print("Failed to create Account Agent runtime")

In [ ]:
# Deploy Orchestrator Agent after specialized agents
print("\n" + "="*60)
print("Deploying Orchestrator Agent to AgentCore Runtime...")
print("="*60)

# First, try to find existing runtime
ORCHESTRATOR_AGENT_ARN = None
ORCHESTRATOR_RUNTIME = None
runtime_name='ecommerce_workshop_orchestrator_notrace'
try:
    response = agentcore_client.list_agent_runtimes()
    # Use 'agentRuntimes' key instead of 'items'
    for runtime in response.get('agentRuntimes', []):
        if runtime.get('agentRuntimeName') == runtime_name:
            print(f"Found existing Orchestrator runtime: {runtime['agentRuntimeId']}")
            ORCHESTRATOR_RUNTIME = {
                'agentRuntimeId': runtime['agentRuntimeId'],
                'agentRuntimeArn': runtime['agentRuntimeArn'],
                'status': runtime.get('status')
            }
            ORCHESTRATOR_AGENT_ARN = runtime['agentRuntimeArn']
            print(f"Using existing Orchestrator ARN: {ORCHESTRATOR_AGENT_ARN}")
            break
except Exception as e:
    print(f"Error checking for existing runtime: {e}")

# If no existing runtime found, create new one
if not ORCHESTRATOR_RUNTIME:
    # Orchestrator environment variables - includes ARNs for specialized agents
    ORCHESTRATOR_ENV = {
        'AGENT_REGION': REGION,
        'MODEL_ID': 'global.anthropic.claude-sonnet-4-5-20250929-v1:0',
        'ORDER_AGENT_ARN': ORDER_AGENT_ARN if ORDER_AGENT_ARN else '',
        'PRODUCT_AGENT_ARN': PRODUCT_AGENT_ARN if PRODUCT_AGENT_ARN else '',
        'ACCOUNT_AGENT_ARN': ACCOUNT_AGENT_ARN if ACCOUNT_AGENT_ARN else ''
    }
    
    ORCHESTRATOR_RUNTIME = create_agent_runtime(
        agentcore_client,
        runtime_name=runtime_name,
        role_arn=RUNTIME_ROLE_ARN,
        container_uri=CONTAINER_URIS.get('orchestrator-agent'),
        environment_vars=ORCHESTRATOR_ENV,
        protocol_type='HTTP',
        description='Main orchestrator agent - routes requests to specialized agents',
        network_mode='PUBLIC'
    )
    
    if ORCHESTRATOR_RUNTIME:
        ORCHESTRATOR_AGENT_ARN = ORCHESTRATOR_RUNTIME['agentRuntimeArn']
        print(f"Created new Orchestrator Runtime ID: {ORCHESTRATOR_RUNTIME['agentRuntimeId']}")
        print(f"Orchestrator Agent ARN: {ORCHESTRATOR_AGENT_ARN}")
    else:
        print("Failed to create Orchestrator runtime")

## Step 15: Test Multi-Agent Orchestration

Test the orchestrator agent to verify it's properly routing to specialized agents and using MCP tools.

In [ ]:
import json
import uuid
import time

def test_orchestrator_agent(query, orchestrator_arn=None, region=None):
    """
    Test the orchestrator agent with a query.
    
    Args:
        query: The user query to test
        orchestrator_arn: ARN of the orchestrator runtime
        region: AWS region
    
    Returns:
        dict: Response from the orchestrator
    """
    if not orchestrator_arn:
        orchestrator_arn = ORCHESTRATOR_AGENT_ARN
    if not region:
        region = REGION
        
    client = boto3.client('bedrock-agentcore', region_name=region)
    
    # Generate a unique session ID (must be at least 33 characters)
    session_id = f"session-{int(time.time())}-{uuid.uuid4().hex[:20]}"
    
    payload = {
        "input": {
            "prompt": query
        }
    }
    
    print(f"🤖 Testing Orchestrator Agent")
    print(f"📍 ARN: {orchestrator_arn}")
    print(f"🔍 Query: {query}")
    print(f"📝 Session ID: {session_id}")
    print("-" * 80)
    
    try:
        # Invoke the orchestrator
        response = client.invoke_agent_runtime(
            agentRuntimeArn=orchestrator_arn,
            runtimeSessionId=session_id,
            payload=json.dumps(payload).encode('utf-8'),
            qualifier="DEFAULT"
        )
        
        # Read response
        response_body = response.get('response', b'')
        if hasattr(response_body, 'read'):
            response_body = response_body.read()
        
        response_data = json.loads(response_body)
        
        # Extract and display response
        if 'output' in response_data:
            output = response_data['output']
            
            # Display agent info if available
            if isinstance(output, dict):
                if 'agent' in output:
                    print(f"✅ Response from: {output['agent']}")
                if 'tools_used' in output:
                    print(f"🔧 Tools used: {output['tools_used']}")
                if 'timestamp' in output:
                    print(f"⏰ Timestamp: {output['timestamp']}")
                print("-" * 80)
                
                # Display the message
                if 'message' in output:
                    print("📨 Response:")
                    print(output['message'])
                else:
                    print("📨 Response:")
                    print(json.dumps(output, indent=2))
            else:
                print("📨 Response:")
                print(output)
        else:
            print("❌ Unexpected response format:")
            print(json.dumps(response_data, indent=2))
            
        return response_data
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return {"error": str(e)}

# Test 1: Order Status Query
print("=" * 80)
print("TEST 1: ORDER STATUS QUERY")
print("=" * 80)
test_orchestrator_agent("What is the status of order ORD-2024-10001?")
print("\n")

# Test 2: Product Search Query
print("=" * 80)
print("TEST 2: PRODUCT SEARCH QUERY")
print("=" * 80)
test_orchestrator_agent("Do you have any wireless headphones under $100?")
print("\n")

# Test 3: Multi-Domain Query
print("=" * 80)
print("TEST 3: MULTI-DOMAIN QUERY")
print("=" * 80)
test_orchestrator_agent("Show me the status of order ORD-2024-10002 and also what wireless products you have under $150")
print("\n")

# Test 4: Account Query
print("=" * 80)
print("TEST 4: ACCOUNT QUERY")
print("=" * 80)
test_orchestrator_agent("What are the benefits of Gold membership?")
print("\n")

print("=" * 80)
print("✅ ORCHESTRATOR TESTING COMPLETE")
print("=" * 80)
print(f"Orchestrator ARN: {ORCHESTRATOR_AGENT_ARN}")
print(f"Order Agent ARN: {ORDER_AGENT_ARN}")
print(f"Product Agent ARN: {PRODUCT_AGENT_ARN}")
print(f"Account Agent ARN: {ACCOUNT_AGENT_ARN}")

## Step 17: Save Configuration for Streamlit App

In [ ]:
import json
import uuid
import time

def test_orchestrator_agent(query, orchestrator_arn=None, region=None):
    """
    Test the orchestrator agent with a query.
    
    Args:
        query: The user query to test
        orchestrator_arn: ARN of the orchestrator runtime
        region: AWS region
    
    Returns:
        dict: Response from the orchestrator
    """
    if not orchestrator_arn:
        orchestrator_arn = ORCHESTRATOR_AGENT_ARN
    if not region:
        region = REGION
        
    client = boto3.client('bedrock-agentcore', region_name=region)
    
    # Generate a unique session ID (must be at least 33 characters)
    session_id = f"session-{int(time.time())}-{uuid.uuid4().hex[:20]}"
    
    payload = {
        "input": {
            "prompt": query
        }
    }
    
    print(f"🤖 Testing Orchestrator Agent")
    print(f"📍 ARN: {orchestrator_arn}")
    print(f"🔍 Query: {query}")
    print(f"📝 Session ID: {session_id}")
    print("-" * 80)
    
    try:
        # Invoke the orchestrator
        response = client.invoke_agent_runtime(
            agentRuntimeArn=orchestrator_arn,
            runtimeSessionId=session_id,
            payload=json.dumps(payload).encode('utf-8'),
            qualifier="DEFAULT"
        )
        
        # Read response
        response_body = response.get('response', b'')
        if hasattr(response_body, 'read'):
            response_body = response_body.read()
        
        response_data = json.loads(response_body)
        
        # Extract and display response
        if 'output' in response_data:
            output = response_data['output']

            # Display agent info if available
            if isinstance(output, dict):
                if 'agent' in output:
                    print(f"✅ Agent: {output['agent']}")
                if 'routed_to' in output:
                    print(f"🔀 Routed to: {', '.join(output.get('routed_to', []))}")
                if 'tools_used' in output:
                    print(f"🔧 Total tools used: {output['tools_used']}")
                if 'routing_details' in output:
                    print(f"📋 Routing details:")
                    for detail in output.get('routing_details', []):
                        print(f"   - {detail.get('agent', 'unknown')}: {detail.get('tools_used', 0)} tools")
                if 'timestamp' in output:
                    print(f"⏰ Timestamp: {output['timestamp']}")
                print("-" * 80)

                # Display the message
                if 'message' in output:
                    print("📨 Response:")
                    print(output['message'])
                else:
                    print("📨 Response:")
                    print(json.dumps(output, indent=2))
            else:
                print("📨 Response:")
                print(output)
        else:
            print("❌ Unexpected response format:")
            print(json.dumps(response_data, indent=2))
            
        return response_data
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return {"error": str(e)}

# Test 1: Order Status Query
print("=" * 80)
print("TEST 1: ORDER STATUS QUERY")
print("=" * 80)
test_orchestrator_agent("What is the status of order ORD-2024-10001?")
print("\n")

# Test 2: Product Search Query
print("=" * 80)
print("TEST 2: PRODUCT SEARCH QUERY")
print("=" * 80)
test_orchestrator_agent("Do you have any wireless headphones under $100?")
print("\n")

# Test 3: Multi-Domain Query
print("=" * 80)
print("TEST 3: MULTI-DOMAIN QUERY")
print("=" * 80)
test_orchestrator_agent("Show me the status of order ORD-2024-10002 and also what wireless products you have under $150")
print("\n")

# Test 4: Account Query
print("=" * 80)
print("TEST 4: ACCOUNT QUERY")
print("=" * 80)
test_orchestrator_agent("What are the benefits of Gold membership?")
print("\n")

print("=" * 80)
print("✅ ORCHESTRATOR TESTING COMPLETE")
print("=" * 80)
print(f"Orchestrator ARN: {ORCHESTRATOR_AGENT_ARN}")
print(f"Order Agent ARN: {ORDER_AGENT_ARN}")
print(f"Product Agent ARN: {PRODUCT_AGENT_ARN}") 
print(f"Account Agent ARN: {ACCOUNT_AGENT_ARN}")

In [ ]:
# Save updated configuration with Orchestrator Agent
if 'ORCHESTRATOR_AGENT_ARN' in locals() and ORCHESTRATOR_RUNTIME:
    orchestrator_config = {
        'orchestrator_arn': ORCHESTRATOR_AGENT_ARN,
        'region': REGION,
        'gateway_url': GATEWAY_URL,
        'gateway_id': GATEWAY_ID,
        'auth': {
            'user_pool_id': USER_POOL_ID,
            'client_id': CLIENT_ID,
            'client_secret': CLIENT_SECRET
        }
    }
    
    # Add specialized agents if they exist
    if 'ORDER_AGENT_ARN' in locals() and ORDER_RUNTIME:
        orchestrator_config['order_agent'] = {
            'arn': ORDER_AGENT_ARN,
            'runtime_id': ORDER_RUNTIME['agentRuntimeId']
        }
    
    if 'PRODUCT_AGENT_ARN' in locals() and PRODUCT_RUNTIME:
        orchestrator_config['product_agent'] = {
            'arn': PRODUCT_AGENT_ARN,
            'runtime_id': PRODUCT_RUNTIME['agentRuntimeId']
        }
    
    if 'ACCOUNT_AGENT_ARN' in locals() and ACCOUNT_RUNTIME:
        orchestrator_config['account_agent'] = {
            'arn': ACCOUNT_AGENT_ARN,
            'runtime_id': ACCOUNT_RUNTIME['agentRuntimeId']
        }
    
    orchestrator_config['orchestrator'] = {
        'arn': ORCHESTRATOR_AGENT_ARN,
        'runtime_id': ORCHESTRATOR_RUNTIME['agentRuntimeId']
    }
    
    # Save for Streamlit app
    save_gateway_config(orchestrator_config, 'streamlit_app/orchestrator_config.json')
    
    print("Configuration saved for Streamlit app:")
    print(f"  Orchestrator ARN: {ORCHESTRATOR_AGENT_ARN}")
    print(f"  Config file: streamlit_app/orchestrator_config.json")
    
    print("\n" + "="*60)
    print("MULTI-AGENT DEPLOYMENT COMPLETE!")
    print("="*60)
    print(f"\nOrchestrator Agent ARN: {ORCHESTRATOR_AGENT_ARN}")
    if 'ORDER_AGENT_ARN' in locals():
        print(f"Order Agent ARN: {ORDER_AGENT_ARN}")
    if 'PRODUCT_AGENT_ARN' in locals():
        print(f"Product Agent ARN: {PRODUCT_AGENT_ARN}")
    if 'ACCOUNT_AGENT_ARN' in locals():
        print(f"Account Agent ARN: {ACCOUNT_AGENT_ARN}")
    print(f"\nGateway URL (MCP Tools): {GATEWAY_URL}")
    print(f"\nTo test the agents:")
    print(f"  Use AWS CLI: aws bedrock-agentcore invoke-agent-runtime")
    print(f"  Or launch the Streamlit app:")
    print(f"    cd streamlit_app")
    print(f"    pip install streamlit")
    print(f"    streamlit run app.py")
    print(f"\nThe app will be available at http://localhost:8501")
    print("="*60)
else:
    print("Note: Some agents failed to deploy. Please check the errors above.")

## Step 18: Launch Streamlit App

Run the Streamlit chat interface to interact with the production agent.

```bash
cd streamlit_app
pip install streamlit
streamlit run app.py
```

---

## Module 3 Complete!

You have successfully deployed the e-commerce multi-agent system to production using Amazon Bedrock AgentCore.

### What Was Created

| Component | Description |
|-----------|-------------|
| **Lambda Functions** | 3 functions (order, product, account tools) |
| **Cognito User Pool** | JWT authentication for Gateway access |
| **AgentCore Gateway** | MCP protocol endpoint with 17 tools |
| **ECR Repositories** | 4 container images for agents |
| **Order Agent Runtime** | Specialized agent for order management |
| **Product Agent Runtime** | Specialized agent for product catalog |
| **Account Agent Runtime** | Specialized agent for account management |
| **Orchestrator Runtime** | Main entry point routing via A2A |
| **Streamlit App** | Interactive chat UI |

### Architecture Benefits

- **Multi-Agent**: Specialized agents for different domains
- **A2A Protocol**: Inter-agent communication
- **Scalable**: Serverless auto-scaling
- **Secure**: JWT authentication with Cognito
- **Standard Protocol**: MCP for tool orchestration

### Files Created

| File | Description |
|------|-------------|
| `lambda_tools/order_tools_lambda.py` | Order management Lambda (5 tools) |
| `lambda_tools/product_tools_lambda.py` | Product catalog Lambda (6 tools) |
| `lambda_tools/account_tools_lambda.py` | Account management Lambda (6 tools) |
| `agents/order_agent.py` | Order Agent A2A server |
| `agents/product_agent.py` | Product Agent A2A server |
| `agents/account_agent.py` | Account Agent A2A server |
| `agents/orchestrator_agent.py` | Orchestrator Agent A2A server |
| `agents/Dockerfile` | Container build file |
| `streamlit_app/app.py` | Chat UI with A2A support |

### Next Steps

1. Test the Streamlit app with various customer queries
2. Monitor agent performance in CloudWatch
3. Add more specialized agents as needed
4. Configure production authentication flows